# 16. Ordered Structures: When Hashing Is Not Enough

A hash map gives O(1) lookup, but it knows **nothing about order or closeness**. Questions like "the next larger value", "the closest value", "the k-th smallest" need **ordered** structures.

**Learning order:** WHAT HASHING CANNOT DO → BISECT ON A SORTED LIST → FLOOR AND CEILING → BUCKETS FOR NEARBY VALUES → LAZY DELETION IN A HEAP → CHOOSING → INTERVIEW THINKING

## 1. What hashing cannot do

| Need | Hash map / set | Ordered structure |
|---|---|---|
| exact lookup | **O(1)** | O(log n) |
| smallest / largest | O(n) | **O(1)** or O(log n) |
| next larger / smaller (successor, predecessor) | O(n) | **O(log n)** |
| values in a range `[a, b]` | O(n) | O(log n + k) |
| k-th smallest | O(n) | O(log n) with a tree |
| iterate in sorted order | O(n log n) sort | O(n) |

Python has no built-in balanced tree (Java has `TreeMap`, C++ has `std::map`). Options: a **sorted list + `bisect`** (insert is O(n) but very fast in practice), the third-party `sortedcontainers.SortedList` (O(log n)), or a **heap** when you only need the min/max.

## 2. Floor and ceiling with `bisect`

- **Ceiling** of `x` = smallest element `>= x` → `bisect_left`.
- **Floor** of `x` = largest element `<= x` → `bisect_right - 1`.

In [1]:
from bisect import bisect_left, bisect_right, insort

def floor_ceiling(sorted_list, x):
    i = bisect_right(sorted_list, x) - 1
    floor = sorted_list[i] if i >= 0 else None
    j = bisect_left(sorted_list, x)
    ceiling = sorted_list[j] if j < len(sorted_list) else None
    return floor, ceiling

data = [1, 3, 5, 7]
print(floor_ceiling(data, 4), floor_ceiling(data, 0), floor_ceiling(data, 8), floor_ceiling(data, 5))   # (3, 5) (None, 1) (7, None) (5, 5)

insort(data, 4)                              # keeps the list sorted, O(n) shift
print(data)   # [1, 3, 4, 5, 7]

(3, 5) (None, 1) (7, None) (5, 5)
[1, 3, 4, 5, 7]


## 3. Contains Duplicate III: buckets turn "close" into "same or adjacent"

Is there a pair with `|nums[i] - nums[j]| <= t` and `|i - j| <= k`? A sorted window works (O(n log k)). A **bucket hash map** gives O(n):

- Bucket width `t + 1`; value `x` goes to bucket `x // (t + 1)`.
- Two values in the **same bucket** differ by at most `t`. Values in **adjacent buckets** might, so check their stored values.
- Keep only the last `k` elements (remove the one leaving the window).

In [2]:
def contains_nearby_almost_duplicate(nums, k, t):
    if t < 0:
        return False
    width, buckets = t + 1, {}
    for i, x in enumerate(nums):
        b = x // width
        if b in buckets:
            return True
        if b - 1 in buckets and x - buckets[b - 1] <= t:
            return True
        if b + 1 in buckets and buckets[b + 1] - x <= t:
            return True
        buckets[b] = x
        if i >= k:
            del buckets[nums[i - k] // width]
    return False

print(contains_nearby_almost_duplicate([1, 2, 3, 1], 3, 0), contains_nearby_almost_duplicate([1, 0, 1, 1], 1, 2), contains_nearby_almost_duplicate([1, 5, 9, 1, 5, 9], 2, 3))   # True True False

True True False


## 4. Heap with lazy deletion (delete an arbitrary item)

A heap cannot delete an arbitrary element in O(log n). **Lazy deletion:** record the removal in a counter and **skip** removed items when they reach the top.

In [3]:
import heapq
from collections import Counter

class RemovableMinHeap:
    def __init__(self):
        self.heap, self.removed, self.size = [], Counter(), 0

    def push(self, x):
        heapq.heappush(self.heap, x)
        self.size += 1

    def remove(self, x):
        self.removed[x] += 1
        self.size -= 1

    def _clean(self):
        while self.heap and self.removed[self.heap[0]] > 0:
            self.removed[heapq.heappop(self.heap)] -= 1

    def min(self):
        self._clean()
        return self.heap[0]

h = RemovableMinHeap()
for x in [5, 2, 8, 2]:
    h.push(x)
h.remove(2)
print(h.min())    # 2   (one copy of 2 is left)
h.remove(2)
print(h.min(), h.size)   # 5 2

2
5 2


## 5. Choosing

| Need | Choose |
|---|---|
| exact membership/counts | `dict` / `set` / `Counter` |
| repeatedly the smallest or largest | `heapq` |
| floor / ceiling / range, few inserts | sorted list + `bisect` |
| floor / ceiling with many inserts and deletes | `sortedcontainers.SortedList` or a balanced BST |
| nearby values (difference ≤ t) | bucket hashing |
| order of insertion / recency | `OrderedDict` / `deque` |

## 6. Interview thinking

**Say:** "A hash map cannot answer successor queries efficiently, so I keep the values in a sorted list and use binary search, O(log n) lookup and O(n) insert. If inserts dominate, I'd use a balanced tree or `SortedList`."

**Common mistakes:** using a dict when the question is about ordering/closeness, forgetting that `insort` is O(n), floor/ceiling off-by-one (`bisect_left` vs `bisect_right`), forgetting to remove the element that left the window in the bucket solution.

## Quick Revision

- Hash = exact lookup only. Ordered questions need sorted lists, heaps or trees.
- Ceiling = `bisect_left`, floor = `bisect_right - 1`. `insort` is O(n).
- Nearby values: bucket by `x // (t + 1)` and check adjacent buckets.
- Heap arbitrary delete: lazy deletion with a counter.